### Deuxieme methode : keras + Embedding

In [ ]:
# Basic packages
import pandas as pd 
import numpy as np
import re
import collections
import matplotlib.pyplot as plt
from pathlib import Path

# Packages for data preparation
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

# Packages for modeling
from keras import models
from keras import layers
from keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from src.preprocessing import preprocessing, split_data


In [ ]:
df = preprocessing("pcm")
X_train, y_train, X_val,y_val,X_test, y_test = split_data(df)

In [ ]:
NB_WORDS = 100000  # Parameter indicating the number of words we'll put in the dictionary
VAL_SIZE = 1000  # Size of the validation set
NB_START_EPOCHS = 100  # Number of epochs we usually start to train with
BATCH_SIZE = 512  # Size of the batches used in the mini-batch gradient descent
MAX_LEN = 145  # Maximum number of words in a sequence
GLOVE_DIM = 50  # Number of dimensions of the GloVe word embeddings
INPUT_PATH = '../input'  # Path where all input files are stored

### Some helper functions


In [ ]:
def deep_model(model, X_train, y_train, X_valid, y_valid):
    '''
    Function to train a multi-class model. The number of epochs and 
    batch_size are set by the constants at the top of the
    notebook. 
    
    Parameters:
        model : model with the chosen architecture
        X_train : training features
        y_train : training target
        X_valid : validation features
        Y_valid : validation target
    Output:
        model training history
    '''

    early_stop = EarlyStopping(
    monitor='val_loss',     # Surveille la perte sur le jeu de validation
    patience=3,             # Nombre d'époques à attendre sans amélioration avant d'arrêter
    restore_best_weights=True # Restaure automatiquement les meilleurs poids à la fin
)

    model.compile(optimizer='rmsprop'
                  , loss='sparse_categorical_crossentropy'
                  , metrics=['accuracy']
                  )
    
    history = model.fit(X_train
                       , y_train
                       , epochs=NB_START_EPOCHS
                       , batch_size=BATCH_SIZE
                       , validation_data=(X_valid, y_valid)
                       , verbose=1
                       , callbacks = [early_stop])
    return history


def eval_metric(history, metric_name):
    '''
    Function to evaluate a trained model on a chosen metric. 
    Training and validation metric are plotted in a
    line chart for each epoch.
    
    Parameters:
        history : model training history
        metric_name : loss or accuracy
    Output:
        line chart with epochs of x-axis and metric on
        y-axis
    '''
    metric = history.history[metric_name]
    val_metric = history.history['val_' + metric_name]

    e = range(1, NB_START_EPOCHS + 1)

    plt.plot(e, metric, 'bo', label='Train ' + metric_name)
    plt.plot(e, val_metric, 'b', label='Validation ' + metric_name)
    plt.legend()
    plt.show()

def test_model(model, X_train, y_train, X_test, y_test, epoch_stop):
    '''
    Function to test the model on new data after training it
    on the full training data with the optimal number of epochs.
    
    Parameters:
        model : trained model
        X_train : training features
        y_train : training target
        X_test : test features
        y_test : test target
        epochs : optimal number of epochs
    Output:
        test accuracy and test loss
    '''
    model.fit(X_train
              , y_train
              , epochs=epoch_stop
              , batch_size=BATCH_SIZE
              , verbose=0)
    results = model.evaluate(X_test, y_test)
    
    return results




#### Converting words to numbers

In [ ]:
tk = Tokenizer(num_words=NB_WORDS,
               lower=True,
               split=" ")
tk.fit_on_texts(X_train)

X_train_seq = tk.texts_to_sequences(X_train)
X_val_seq = tk.texts_to_sequences(X_val)
X_test_seq = tk.texts_to_sequences(X_test)


#### Creating word sequences of equal length


In [ ]:
seq_lengths = X_train.apply(lambda x: len(x.split(' ')))
seq_lengths.describe()

In [ ]:
X_train_seq_trunc = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_val_seq_trunc = pad_sequences(X_val_seq, maxlen=MAX_LEN)
X_test_seq_trunc = pad_sequences(X_test_seq, maxlen=MAX_LEN)

In [ ]:
X_train_seq_trunc[10]  # Example of padded sequence

#### Splitting off validation data¶


In [ ]:

assert X_val_seq_trunc.shape[0] == y_val.shape[0]
assert X_train_seq_trunc.shape[0] == y_train.shape[0]

print('Shape of validation set:',X_val.shape)


### Modeling

##### Training word embeddings
Keras provides an Embedding layer which helps us to train specific word embeddings based on our training data. It will convert the words in our vocabulary to multi-dimensional vectors.

In [ ]:
emb_model = models.Sequential()
emb_model.add(layers.Input(shape=(MAX_LEN,)))
emb_model.add(layers.Embedding(NB_WORDS, 8))
emb_model.add(layers.Bidirectional(layers.LSTM(64, return_sequences=False)))
emb_model.add(layers.Dropout(0.5))
emb_model.add(layers.Dense(64, activation='relu'))
#emb_model.add(layers.Attention())
emb_model.add(layers.Flatten())
emb_model.add(layers.Dense(3, activation='softmax'))
emb_model.summary()

In [ ]:
emb_history = deep_model(emb_model, X_train_seq_trunc, y_train, X_val_seq_trunc, y_val)


In [ ]:
eval_metric(emb_history, 'macro_f1')


In [ ]:
eval_metric(emb_history, 'loss')


In [ ]:
emb_results = test_model(emb_model, X_train_seq_trunc, y_train, X_test_seq_trunc, y_test, 6)
print('/n')

print(f"Test accuracy of word embeddings model: {emb_results[1] * 100:.2f}%")